# Border basis — analysis notebook

Inspect dataset and (post-training) model outputs for the border-basis task over GF(p).

**Prerequisite**: dataset generated and model trained:
```bash
cd ../experiments/toy/scripts
python generate.py
python train.py
```

*Note: `border_basis` uses the Gröbner basis as a stand-in until SageMath ships a stable border basis.*

Patterns mirror `calt-codebase/examples/demos/minimal_demo.ipynb`.

In [ ]:
import os
import sys
from pathlib import Path

_TASK_ROOT = Path.cwd().parent
_REPO_ROOT = _TASK_ROOT.parent
os.chdir(_TASK_ROOT / 'experiments' / 'toy' / 'scripts')
sys.path.insert(0, str(_REPO_ROOT))

from omegaconf import OmegaConf
from calt.dataset.sagemath.utils.polynomial_sampler import PolynomialSampler

from border_basis.core.generator import BorderBasisGenerator
from shared import showcase

## 1. A few raw samples over GF(7)

In [ ]:
data_cfg = OmegaConf.load('../configs/data.yaml')
sampler = PolynomialSampler(**OmegaConf.to_container(data_cfg.sampler, resolve=True))
gen = BorderBasisGenerator(sampler=sampler, **OmegaConf.to_container(data_cfg.problem_generator, resolve=True))

for seed in range(3):
    G_in, B = gen(seed)
    print(f'seed={seed}')
    print(f'  generators = {[str(g) for g in G_in]}')
    print(f'  basis      = {[str(b) for b in B]}')
    print()

## 2. Stored dataset

In [ ]:
train_path = Path('../data/GF7/train_raw.txt')
if not train_path.exists():
    print(f'No dataset at {train_path.resolve()}. Run scripts/generate.py first.')
else:
    lines = train_path.read_text().strip().split('\n')
    print(f'{len(lines)} train samples. First 3:')
    for line in lines[:3]:
        print(' ', line)

## 3. Showcase eval results — canonical CALT pattern

In [ ]:
from calt.io import IOPipeline

cfg = OmegaConf.load('../configs/train.yaml')
io_dict = IOPipeline.from_config(cfg.data).build()

results_dir = '../outputs/results'
if not Path(results_dir).exists():
    print(f'No results directory at {Path(results_dir).resolve()}. Run train.py first.')
else:
    showcase(io_dict['test_dataset'], success_cases=True,  num_show=5, results_dir=results_dir)
    print()
    showcase(io_dict['test_dataset'], success_cases=False, num_show=5, results_dir=results_dir)

## 4. Learning curve

In [ ]:
from shared.plotting import plot_success_rate
import matplotlib.pyplot as plt

try:
    ax = plot_success_rate(results_dir)
    plt.show()
except FileNotFoundError as e:
    print(e)